# Embedding Responses to Ethanol Torsional Changes

## Setup

In [ ]:
from setup import *

from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from rdkit.Chem import rdMolTransforms
from scipy.spatial.distance import pdist
from torch_geometric.nn import global_mean_pool

## Compute Ethanol Scan

In [ ]:
def grid_sample_angles(n_grid, dihedral_specs):
    """No randomness anywhere. Returns n_grid^2 angle pairs, evenly spaced.
    endpoint=False avoids duplicating 0 and 360 (same angle, periodic)."""
    grid_1d = np.linspace(0, 360, n_grid, endpoint=False)
    T1, T2 = np.meshgrid(grid_1d, grid_1d, indexing='ij')
    angles = np.stack([T1.ravel(), T2.ravel()], axis=1)
    return angles

def embed_conformers_at_angles(angles, dihedral_specs, base_mol):
    """Given an (N, 2) array of angle pairs, builds one conformer per pair
    (unrelaxed except for the set dihedrals) and computes single-point MMFF
    energy at each."""
    mols, energies = [], []
    for a1, a2 in angles:
        mol = Chem.Mol(base_mol)
        conf = mol.GetConformer()
        for atoms, ang in zip(dihedral_specs, (a1, a2)):
            rdMolTransforms.SetDihedralDeg(conf, *atoms, float(ang))
        props = AllChem.MMFFGetMoleculeProperties(mol)
        ff = AllChem.MMFFGetMoleculeForceField(mol, props)
        energies.append(ff.CalcEnergy())  # single-point, no optimization
        mols.append(mol)
    return mols, np.array(energies)

def evaluate_energy_grid(dihedral_specs, base_mol, n_grid=72):
    """Reuses the same relaxed base geometry as the sample grid so the
    surface and the scatter are directly comparable. n_grid points per axis
    (72 -> 5-degree resolution), evaluated on a full regular meshgrid."""
    grid_1d = np.linspace(0, 360, n_grid, endpoint=False)
    T1, T2 = np.meshgrid(grid_1d, grid_1d, indexing='ij')
    E = np.zeros_like(T1)
    for i in range(n_grid):
        for j in range(n_grid):
            mol = Chem.Mol(base_mol)
            conf = mol.GetConformer()
            for atoms, ang in zip(dihedral_specs, (T1[i, j], T2[i, j])):
                rdMolTransforms.SetDihedralDeg(conf, *atoms, float(ang))
            props = AllChem.MMFFGetMoleculeProperties(mol)
            ff = AllChem.MMFFGetMoleculeForceField(mol, props)
            E[i, j] = ff.CalcEnergy()
    return T1, T2, E

n_grid_sample = 64  # 64^2 = 4096 conformers
dihedral_specs = [(3, 0, 1, 2), (0, 1, 2, 8)]

# base geometry (relaxed, before torsions are set) — needed for both the
# sample grid and the surface grid, so build it once and reuse
base_mol = Chem.AddHs(Chem.MolFromSmiles("CCO"))
AllChem.EmbedMolecule(base_mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(base_mol)
angles = grid_sample_angles(n_grid_sample, dihedral_specs)
mols, energies = embed_conformers_at_angles(angles, dihedral_specs, base_mol)
T1, T2, E_grid = evaluate_energy_grid(dihedral_specs, base_mol, n_grid=72)

model_paths = {
    "No Pretraining": CHECKPOINTS["nopre"],
    "Normal NT-Xent": CHECKPOINTS["NT"],
    "Aware NT-Xent": CHECKPOINTS["ANT"],
}
datalist = [mol_to_graph(m, use_3d=True) for m in mols]
embeddings = {}
batch_size = 64
print(f"Encoding {len(mols):,} shared ethanol geometries.")
for name, checkpoint in model_paths.items():
    model = EncoderModel.load_model(checkpoint, device=DEVICE).eval()
    if name == "No Pretraining":
        encoded, pooled = [], []
        with torch.inference_mode():
            for start in tqdm(range(0, len(datalist), batch_size), desc=name):
                batch = Batch.from_data_list(datalist[start:start + batch_size]).to(DEVICE)
                atomic = model.initializer(batch)
                pooled.append(torch.nn.functional.normalize(
                    global_mean_pool(atomic.x, atomic.batch, size=batch.num_graphs),
                    p=2, dim=1).cpu())
                del atomic
                encoded.append(model.encode(batch).cpu())
        embeddings[name] = torch.cat(encoded).float().numpy()
        embeddings["Pooled AIMNet2"] = torch.cat(pooled).float().numpy()
        del batch
    else:
        embeddings[name] = encode_graphs(model, datalist, chunk_size=batch_size).numpy()
    del model

# Match compute_pca: standardize each dimension, then fit a separate PCA.
representations = ["Pooled AIMNet2", "No Pretraining", "Normal NT-Xent", "Aware NT-Xent"]
pca_coordinates, pca_variance = {}, {}
for name in representations:
    standardized = StandardScaler().fit_transform(embeddings[name])
    pca = PCA(n_components=2, svd_solver="full")
    pca_coordinates[name] = pca.fit_transform(standardized)
    pca_variance[name] = 100 * pca.explained_variance_ratio_
    pc1, pc2 = pca_variance[name]
    print(f"{name}: PC1 {pc1:.1f}%, PC2 {pc2:.1f}%")

# ---- 6. Energy-distance agreement in the full embeddings, before PCA ----
# Every unordered geometry pair is used once; the same pairs serve all controls.
energy_differences = pdist(np.asarray(energies, dtype=np.float64)[:, None],
                                   metric="cityblock")
energy_correlations = {}
print(f"Energy-distance Spearman correlations across {len(energy_differences):,} "
      "unordered geometry pairs (full embeddings, before PCA):")
for name in representations:
    pair_distances = pdist(np.asarray(embeddings[name], dtype=np.float64),
                                   metric="cosine")
    energy_correlations[name] = float(spearmanr(
        pair_distances, energy_differences).statistic)
    print(f"{name}: rho = {energy_correlations[name]:.4f}")
    del pair_distances
# Shared geometries make pair observations dependent, so no pair-based p-values are reported.
del energy_differences

## Figure 5: Ethanol Torsional Landscape

In [ ]:
with plt.rc_context({
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 0.65,
    "axes.labelcolor": "#222222",
    "text.color": "#222222",
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "xtick.color": "#222222",
    "ytick.color": "#222222",
}):
    fig = plt.figure(figsize=(17.1 * cm, 10 * cm), dpi=600, layout="constrained")
    fig.get_layout_engine().set(w_pad=0.025, h_pad=0.025, wspace=0.015, hspace=0.045)
    gs = fig.add_gridspec(3, 3, width_ratios=[1.95, 1, 1],
                           height_ratios=[1, 1, 0.10])
    vmin = min(energies.min(), E_grid.min())
    vmax = max(energies.max(), E_grid.max())
    ethanol_color_norm = Normalize(vmin=vmin, vmax=vmax)

    def _style_2d(ax, title, letter, label_offset=-12):
        if len(title) > 20:
            title = "\n".join(title.rsplit(" ", 1))
        ax.set_title(title, pad=8)
        ax.annotate(f"{letter}.)", xy=(0, 1), xycoords="axes fraction", xytext=(label_offset, 8),
                    textcoords="offset points", ha="left", va="bottom",
                    fontsize=10, fontweight="bold", annotation_clip=False)
        ax.tick_params(direction="out", length=2.5, width=0.65, pad=2)
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    ax1 = fig.add_subplot(gs[:2, 0], projection="3d")
    surf = ax1.plot_surface(T1, T2, E_grid, cmap="viridis", norm=ethanol_color_norm,
                            linewidth=0, antialiased=True, rstride=1, cstride=1)
    _style_2d(ax1, "MMFF94 energy", "a", label_offset=0)
    ax1.set(xlim=(0, 360), ylim=(0, 360),
            xticks=[0, 120, 240, 360], yticks=[0, 120, 240, 360])
    ax1.set_xlabel(r"$\theta_{\mathrm{CC}}$ (°)", labelpad=-3)
    ax1.set_ylabel(r"$\theta_{\mathrm{CO}}$ (°)", labelpad=-3)
    ax1.zaxis.set_major_locator(MaxNLocator(nbins=4))
    ax1.tick_params(labelsize=7, pad=-1)
    ax1.view_init(elev=20, azim=-60)
    ax1.set_box_aspect((1, 1, 0.7))
    ax1.set_anchor("E")
    for axis in (ax1.xaxis, ax1.yaxis, ax1.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor((0.7, 0.7, 0.7, 0.4))
        axis.pane.set_linewidth(0.35)
        axis.line.set_linewidth(0.55)
        axis._axinfo["grid"]["linewidth"] = 0.3
        axis._axinfo["grid"]["color"] = (0.7, 0.7, 0.7, 0.35)

    ethanol_axes = [ax1]
    for name, letter, position in zip(representations, "bcde",
                                     [(0, 1), (0, 2), (1, 1), (1, 2)]):
        ax = fig.add_subplot(gs[position])
        coords = pca_coordinates[name]
        lower, upper = coords.min(axis=0), coords.max(axis=0)
        center = (lower + upper) / 2
        half_width = 0.525 * max(upper[0] - lower[0], (upper[1] - lower[1]) / 0.85)
        ax.set_xlim(center[0] - half_width, center[0] + half_width)
        ax.set_ylim(center[1] - 0.85 * half_width, center[1] + 0.85 * half_width)
        ax.scatter(coords[:, 0], coords[:, 1], c=energies, cmap="viridis",
                   norm=ethanol_color_norm, s=1.4, edgecolors="none", rasterized=True)
        pc1, pc2 = pca_variance[name]
        ax.set_xlabel(f"PC1 ({pc1:.1f}%)", labelpad=3)
        ax.set_ylabel(f"PC2 ({pc2:.1f}%)", labelpad=0)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3))
        ax.ticklabel_format(style="sci", axis="both", scilimits=(-2, 3))
        ax.xaxis.get_offset_text().set_fontsize(7)
        ax.yaxis.get_offset_text().set_fontsize(7)
        ax.set_aspect("equal", adjustable="box")
        ax.set_box_aspect(0.75)
        ax.set_anchor("NE" if position[1] == 1 else "NW")
        _style_2d(ax, MODEL_TITLES.get(name, name), letter)
        ethanol_axes.append(ax)

    # A shorter, thicker colorbar keeps the energy scale readable at print size.
    cbar_ax = fig.add_subplot(gs[2, :])
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=ethanol_color_norm, cmap="viridis"),
                        cax=cbar_ax, orientation="horizontal")
    cbar.set_label(r"MMFF94 energy (kcal mol$^{-1}$)", labelpad=4)
    cbar.locator = MaxNLocator(nbins=5)
    cbar.update_ticks()
    cbar.ax.tick_params(length=2.5, width=0.65, pad=2)
    cbar.outline.set_visible(False)
    fig.savefig(OUTPUT_DIR / "ethanol_tsne_energy_landscape.png", dpi=600)
    plt.show()

## PCA Variance and Energy Correlations

In [ ]:
display(pd.DataFrame([
    {
        "Representation": name,
        "PC1 variance (%)": pca_variance[name][0],
        "PC2 variance (%)": pca_variance[name][1],
        "Energy-distance Spearman rho": energy_correlations[name],
    }
    for name in representations
]).round(4))